# Treino da ZicoAI (LoRA) — Google Colab grátis

**Antes de tudo:** menu **Ambiente de execução › Alterar o tipo de ambiente de execução › GPU T4 › Salvar**.

Depois: menu **Ambiente de execução › Executar tudo**. As DUAS perguntas aparecem logo no começo (passo 2), responda e pode deixar rodando:
1. **Permitir acesso ao Google Drive** (é onde tudo fica salvo, até se a aba cair).
2. **Escolher arquivo**: `treino.jsonl` da pasta `ZicoAI\treino` do PC. (Se ele já estiver no Drive, na pasta "ZicoAI Treino", nem pergunta.)

Leva uns 40 a 60 minutos. No fim o Drive recebe só arquivos pequenos (o LoRA, ~100–200 MB); o modelo base o PC baixa direto do Ollama. Deixe a aba aberta. Se cair no meio, o LoRA já treinado fica salvo no Drive e dá pra gerar só o modelo do Ollama de novo.

In [ ]:
# 1) Confere a placa e instala as ferramentas de treino (Unsloth)
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NENHUMA")
assert gpu, "Sem GPU! Menu Ambiente de execução › Alterar o tipo › GPU T4, e rode de novo."
!pip install -q unsloth
print("Ferramentas instaladas.")

In [ ]:
# 2) Drive primeiro (as perguntas ficam todas no começo) e os exemplos de treino
from google.colab import drive, files
import json, random, os, shutil
drive.mount("/content/drive")
destino = "/content/drive/MyDrive/ZicoAI Treino"
os.makedirs(destino, exist_ok=True)
nome = destino + "/treino.jsonl"
if not os.path.exists(nome):
    enviado = files.upload()          # escolha treino.jsonl da pasta ZicoAI\\treino do PC
    nome_local = [n for n in enviado if n.endswith(".jsonl")][0]
    shutil.copy(nome_local, nome)     # guarda no Drive pra próxima vez não precisar enviar
linhas = [json.loads(l) for l in open(nome, encoding="utf-8") if l.strip()]
random.seed(3407); random.shuffle(linhas)
corte = max(1, len(linhas) // 20)          # 5% fica de fora pra prova depois do treino
prova, treino = linhas[:corte], linhas[corte:]
print(len(treino), "exemplos de treino |", len(prova), "separados pra prova")

In [ ]:
# 3) Carrega a base (Qwen3 8B em 4 bits) e liga o LoRA
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(model_name="unsloth/Qwen3-8B", max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

In [ ]:
# 4) Treina (só aprende as respostas, não decora as perguntas)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
textos = [tokenizer.apply_chat_template(l["messages"], tokenize=False, enable_thinking=False) for l in treino]
ds = Dataset.from_dict({"text": textos})
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   warmup_steps=10, num_train_epochs=2, learning_rate=2e-4, logging_steps=10,
                   optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear", seed=3407,
                   output_dir="saida", report_to="none"))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
r = trainer.train()
print("Treino pronto em %.1f min" % (r.metrics["train_runtime"] / 60))
# salva o LoRA no Drive JÁ (se a aba cair depois, o treino não se perde)
model.save_pretrained("zicoai-lora"); tokenizer.save_pretrained("zicoai-lora")
shutil.make_archive(destino + "/zicoai-lora", "zip", "zicoai-lora")
print("LoRA salvo no Drive:", destino + "/zicoai-lora.zip")

In [ ]:
# 5) Prova rápida: perguntas que ele NUNCA viu no treino
FastLanguageModel.for_inference(model)
res = []
for l in prova[:8]:
    m = l["messages"][:2]
    ids = tokenizer.apply_chat_template(m, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=300, temperature=0.7, top_p=0.8, top_k=20)
    resp = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).replace("<think>", "").replace("</think>", "").strip()
    res.append({"aula": l["aula"], "pergunta": m[1]["content"], "esperado": l["messages"][2]["content"], "zicoai": resp})
    print("❓", m[1]["content"], "\n🤖", resp, "\n")
json.dump(res, open(destino + "/prova_resultado.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# 6) LoRA no formato do Ollama (arquivo pequeno, cabe no Drive). No PC: FROM qwen3:8b + ADAPTER
import glob, subprocess
if not os.path.exists("llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp"], check=True)
!pip install -q gguf
!python llama.cpp/convert_lora_to_gguf.py zicoai-lora --base-model-id Qwen/Qwen3-8B --outtype f16 --outfile zicoai-lora.gguf
assert os.path.exists("zicoai-lora.gguf"), "A conversão do LoRA falhou (veja o erro acima)"
shutil.copy("zicoai-lora.gguf", destino + "/zicoai-lora.gguf")
print("Tamanho: %.0f MB" % (os.path.getsize("zicoai-lora.gguf") / 1e6))
livre = shutil.disk_usage("/content/drive").free / 1e9
print("Drive livre: %.1f GB" % livre)
if livre > 6.5:
    # tem espaço: salva também o modelo inteiro pronto (5 GB), que não depende do base do Ollama
    model.save_pretrained_gguf("zicoai_gguf", tokenizer, quantization_method="q4_k_m")
    q4 = [g for g in glob.glob("zicoai_gguf*/*.gguf") + glob.glob("*.gguf") if "q4_k_m" in g.lower()]
    if q4: shutil.copy(q4[0], destino + "/zicoai-treinada-q4_k_m.gguf")
print("PRONTO! No Drive, pasta 'ZicoAI Treino': zicoai-lora.gguf (pro Ollama), zicoai-lora.zip, prova_resultado.json")